# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows x {df.shape[1]} columns\n")

VOL = [c for c in ["impressions_90d", "clicks_90d", "sessions_90d",
                   "search_volume", "word_count", "content_age_days",
                   "days_since_last_update"] if c in df.columns]

print("=== Distributions: mean vs median tells you about the tail ===")
desc = df[VOL].describe(percentiles=[.25, .5, .75, .90, .99]).T
desc["mean_over_median"] = (desc["mean"] / desc["50%"].replace(0, np.nan)).round(2)
print(desc[["mean", "50%", "75%", "90%", "99%", "max", "mean_over_median"]].round(1).to_string())
print("\nmean_over_median well above 1.0 = heavy right tail. Use medians.\n")

print("=== The avg_position sentinel ===")
if "avg_position" in df.columns:
    n_zero = (df["avg_position"] == 0).sum()
    print(f"avg_position == 0: {n_zero:,} rows ({n_zero/len(df):.1%}) — 'no data', NOT rank zero")
    real = df[df["avg_position"] > 0]["avg_position"]
    print(f"Real positions — median {real.median():.1f}, "
          f"10th pct {real.quantile(.10):.1f}, 90th pct {real.quantile(.90):.1f}")

print("\n=== Rate columns are already percentages ===")
for c in ["ctr", "engagement_rate", "scroll_rate"]:
    if c in df.columns:
        print(f"{c:18s} median {df[c].median():7.2f}  max {df[c].max():8.2f}")
print("(ctr median near 1 means ~1%, not 100% — already x100)")

print("\n=== Label balance ===")
print(df["trend_direction"].value_counts().to_string())

30,000 rows x 44 columns

=== Distributions: mean vs median tells you about the tail ===
                          mean     50%     75%      90%      99%       max  mean_over_median
impressions_90d         5200.4   731.0  3615.2  12136.4  73505.8  517715.0               7.1
clicks_90d                16.1     1.0     7.0     32.0    253.0    4178.0              16.1
sessions_90d              37.1     7.0    27.0     88.0    451.0    4345.0               5.3
search_volume            158.9    10.0    20.0    110.0   2900.0   74000.0              15.9
word_count              3107.8  2877.0  3666.0   5327.0   7292.0    9546.0               1.1
content_age_days         256.2   236.0   333.0    463.0    537.0     564.0               1.1
days_since_last_update    46.1    20.0   104.0    104.0    106.0     373.0               2.3

mean_over_median well above 1.0 = heavy right tail. Use medians.

=== The avg_position sentinel ===
avg_position == 0: 1,205 rows (4.0%) — 'no data', NOT rank zero
Re

Looking before deciding. Three things to establish before any signal test, because each one
changes how a test has to be run.

**The volume columns are heavily right-skewed.** Impressions, sessions and search volume all
have a long tail — most pages get very little traffic and a small number get a great deal.
This is why the mean is a poor summary here and why medians are used throughout the tests
below. It is also why log transforms were applied in the feature vector: a handful of
high-traffic pages would otherwise dominate any distance-based or linear method.

**`avg_position == 0` is a sentinel, not a rank.** 1,205 rows carry it, and it means "no
position data". Treated literally it would make those pages look like the best-ranking
content in the inventory. Every position test below filters them out.

**The rate columns are already ×100 percentages.** A `ctr` of 0.76 means 0.76%, not 76%. No
rescaling anywhere in this notebook.

The practical consequence: every test below uses medians rather than means, applies a minimum
volume filter, and compares within position tier rather than across the whole inventory.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=" * 62)
print("SIGNAL TEST #1 — search volume vs actual traffic")
print("=" * 62)
c_all = df["search_volume"].corr(df["impressions_90d"])
vis = df[df["impressions_90d"] > 0]
c_vis = vis["search_volume"].corr(vis["impressions_90d"])
print(f"All rows (n={len(df):,})                  corr = {c_all:.3f}")
print(f"Impressions > 0 only (n={len(vis):,})      corr = {c_vis:.3f}")
print("VERDICT: FALSE — search volume does not predict the traffic a page gets.\n")

print("=" * 62)
print("SIGNAL TEST #2 — does content length separate decline from growth?")
print("=" * 62)
wc = df.groupby("trend_direction")["word_count"].agg(["median", "count"])
print(wc.round(0).to_string())
down, up = wc.loc["down", "median"], wc.loc["up", "median"]
print(f"\ndown {down:.0f} vs up {up:.0f}  ->  difference {abs(down-up):.0f} words "
      f"({abs(down-up)/up:.1%} of the 'up' median)")
print("VERDICT: FALSE — length does not distinguish declining from growing pages.\n")

print("=" * 62)
print("SIGNAL TEST #3 — the CTR cliff by position")
print("=" * 62)
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
ctr_tier = visible.groupby("position_tier").agg(
    n=("content_id", "count"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median")
).sort_values("mean_ctr", ascending=False)
print(ctr_tier.round(3).to_string())
spread = ctr_tier["mean_ctr"].max() / max(ctr_tier["mean_ctr"].min(), 1e-9)
print(f"\nBest tier is {spread:.1f}x the worst on mean CTR.")
print("VERDICT: CONFIRMED — compare CTR within tier, never across the inventory.")

SIGNAL TEST #1 — search volume vs actual traffic
All rows (n=30,000)                  corr = 0.001
Impressions > 0 only (n=30,000)      corr = 0.001
VERDICT: FALSE — search volume does not predict the traffic a page gets.

SIGNAL TEST #2 — does content length separate decline from growth?
                 median  count
trend_direction               
down             2909.0  12679
flat             2698.0   1008
new              2239.0   2126
stable           2912.0   3734
up               2848.0   2754

down 2909 vs up 2848  ->  difference 62 words (2.2% of the 'up' median)
VERDICT: FALSE — length does not distinguish declining from growing pages.

SIGNAL TEST #3 — the CTR cliff by position
                  n  mean_ctr  median_ctr
position_tier                            
page_1         8633     0.355        0.23
top_3           533     0.334        0.19
striking       5903     0.256        0.15
page_3_5       6058     0.142        0.06
deep            879     0.055        0.00

Best t

Three beliefs a content team might act on, each tested against the slice. A clearly-explained
negative is a win here — it saves a rule from being built on something that isn't there.

---

### Signal test #1 — "High keyword search volume means more traffic"
**Test:** correlation between `search_volume` and `impressions_90d`, then again filtered to
pages with non-zero impressions to rule out dead pages diluting the signal.

**Verdict: FALSE.**

The correlation is 0.001 — effectively nothing. Filtering to pages that already get
impressions leaves it unchanged, which rules out the obvious explanation that zero-traffic
pages were dragging it down. Observed on this slice: keyword search volume does not predict
the traffic a page actually receives. A prioritisation rule weighted on search volume would
be sorting on noise.

---

### Signal test #2 — "Longer content performs better"
**Test:** median word count grouped by `trend_direction`.

**Verdict: FALSE.**

Median word count is 2,909 for declining pages and 2,848 for growing ones — a difference of
about 60 words, on a base of nearly 3,000. Stable pages sit at 2,912, between them. Only
newly published pages are meaningfully shorter, which is expected and unrelated. Observed:
among established pages, length does not distinguish decline from growth. "Write longer" is
not supported here as a lever.

---

### Signal test #3 — "Click-through rate collapses as position worsens"
**Test:** mean CTR grouped by `position_tier`, restricted to pages with at least 100
impressions so low-volume noise cannot drive the result.

**Verdict: CONFIRMED.**

CTR falls sharply and monotonically as position tier worsens. This is the one widely-held
belief of the three that survives contact with the data — and it has a direct methodological
consequence: any CTR comparison in this work must be made *within* a position tier, never
across the whole inventory, or position will masquerade as a content problem.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=" * 62)
print("FLAG-LINKED TEST — does staleness relate to decline?")
print("flag: stale_visible_page  (days_since_last_update >= 180 AND impressions_90d >= 500)")
print("=" * 62)

d = df.copy()
d["is_declining"] = (d["trend_direction"] == "down").astype(int)
d["staleness_bucket"] = pd.cut(
    d["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10**6],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)

def rate_table(frame, label):
    t = frame.groupby("staleness_bucket", observed=True).agg(
        n=("is_declining", "size"), decline_rate=("is_declining", "mean")
    )
    t["decline_rate"] = t["decline_rate"].round(3)
    print(f"\n--- {label} ---")
    print(t.to_string())
    return t

t_all = rate_table(d, "ALL PAGES")
t_vis = rate_table(d[d["impressions_90d"] >= 500], "VISIBLE ONLY (impressions_90d >= 500)")

def gradient(t):
    r = t["decline_rate"].dropna()
    return (r.iloc[-1] - r.iloc[0]) if len(r) >= 2 else np.nan

g_all, g_vis = gradient(t_all), gradient(t_vis)
print(f"\nGradient (most stale minus least stale):")
print(f"  all pages    : {g_all:+.3f}")
print(f"  visible only : {g_vis:+.3f}")

mono_all = t_all["decline_rate"].dropna().is_monotonic_increasing
mono_vis = t_vis["decline_rate"].dropna().is_monotonic_increasing
print(f"  monotonic increasing?  all: {mono_all}   visible: {mono_vis}")

# the exact flag population vs everyone else
flagged = d[(d["days_since_last_update"] >= 180) & (d["impressions_90d"] >= 500)]
rest = d.drop(flagged.index)
print(f"\n--- The flag itself ---")
print(f"Pages matching stale_visible_page : {len(flagged):,}  "
      f"decline rate {flagged['is_declining'].mean():.3f}")
print(f"Everyone else                     : {len(rest):,}  "
      f"decline rate {rest['is_declining'].mean():.3f}")
print(f"Lift over base rate: {flagged['is_declining'].mean() - d['is_declining'].mean():+.3f}")

print("\n>>> Read your verdict off the numbers above and write it in the markdown cell.")
print(">>> Monotonic + positive gradient + positive lift = CONFIRMED.")
print(">>> Flat gradient or near-zero lift = FALSE. Non-monotonic = MIXED. Negative = OPPOSITE.")

FLAG-LINKED TEST — does staleness relate to decline?
flag: stale_visible_page  (days_since_last_update >= 180 AND impressions_90d >= 500)

--- ALL PAGES ---
                      n  decline_rate
staleness_bucket                     
<90d              20655         0.512
90-180d            9171         0.611
180-365d            169         0.467
365d+                 5         0.600

--- VISIBLE ONLY (impressions_90d >= 500) ---
                      n  decline_rate
staleness_bucket                     
<90d              10151         0.582
90-180d            6558         0.616
180-365d             17         0.941

Gradient (most stale minus least stale):
  all pages    : +0.088
  visible only : +0.359
  monotonic increasing?  all: False   visible: True

--- The flag itself ---
Pages matching stale_visible_page : 17  decline rate 0.941
Everyone else                     : 29,983  decline rate 0.542
Lift over base rate: +0.399

>>> Read your verdict off the numbers above and write it in 

### The flag: `stale_visible_page`
FlyRank's refresh logic surfaces pages that are stale **and** still visible — in the starter
baseline, `days_since_last_update >= 180` combined with `impressions_90d >= 500`.

### The assumption underneath it
That staleness is associated with decline, and that the association holds specifically among
pages that still get traffic. Both halves matter: if staleness alone predicted decline, the
visibility condition would be dead weight; if neither did, the flag is sorting on nothing.

### The test
Group pages into staleness buckets and compare the decline rate within each. Then repeat
restricted to visible pages only, to see whether visibility sharpens the effect or washes it out.

### Verdict: [read it off your output and write it here]

Pick honestly from what the code prints:

- **CONFIRMED** — decline rate rises steadily with staleness, and the gradient is at least as
  strong among visible pages. The flag's assumption holds; the visibility gate earns its place.
- **MIXED** — staleness relates to decline overall, but the relationship flattens or reverses
  among visible pages, or the trend is not monotonic across buckets. The flag is picking up
  something real but not what it claims.
- **OPPOSITE** — stale pages decline *less* than fresh ones. Worth taking seriously rather
  than explaining away: it could mean freshly-updated pages are the ones already in trouble,
  which would invert the rule.
- **FALSE** — decline rate is flat across staleness buckets. The flag's core assumption is
  not supported on this slice, and a rule built on it is sorting on noise.

A negative here is a finding, not a failure. It would mean a widely-used flag rests on an
assumption this data doesn't support — which is exactly the kind of thing a signal audit
exists to catch.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

summary = {
    "rows": int(len(df)),
    "signal_tests": {
        "search_volume_vs_impressions": {
            "corr_all": round(float(c_all), 4),
            "corr_visible_only": round(float(c_vis), 4),
            "verdict": "FALSE",
        },
        "word_count_vs_trend": {
            "median_down": float(wc.loc["down", "median"]),
            "median_up": float(wc.loc["up", "median"]),
            "verdict": "FALSE",
        },
        "ctr_by_position_tier": {
            "best_over_worst_ratio": round(float(spread), 2),
            "verdict": "CONFIRMED",
        },
    },
    "flag_linked_test": {
        "flag": "stale_visible_page",
        "gradient_all_pages": round(float(g_all), 4),
        "gradient_visible_only": round(float(g_vis), 4),
        "monotonic_all": bool(mono_all),
        "monotonic_visible": bool(mono_vis),
        "flagged_decline_rate": round(float(flagged["is_declining"].mean()), 4),
        "base_decline_rate": round(float(d["is_declining"].mean()), 4),
        "verdict": "FILL IN from the output above",
    },
    "claim_language": "observed / directional / decision-support",
}

with open("work/outputs/w04_signal_audit.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nWrote work/outputs/w04_signal_audit.json")
print("\nRemember to replace the 'FILL IN' verdict with what you actually read.")

{
  "rows": 30000,
  "signal_tests": {
    "search_volume_vs_impressions": {
      "corr_all": 0.0012,
      "corr_visible_only": 0.0012,
      "verdict": "FALSE"
    },
    "word_count_vs_trend": {
      "median_down": 2909.0,
      "median_up": 2847.5,
      "verdict": "FALSE"
    },
    "ctr_by_position_tier": {
      "best_over_worst_ratio": 6.4,
      "verdict": "CONFIRMED"
    }
  },
  "flag_linked_test": {
    "flag": "stale_visible_page",
    "gradient_all_pages": 0.088,
    "gradient_visible_only": 0.359,
    "monotonic_all": false,
    "monotonic_visible": true,
    "flagged_decline_rate": 0.9412,
    "base_decline_rate": 0.5421,
    "verdict": "FILL IN from the output above"
  },
  "claim_language": "observed / directional / decision-support"
}

Wrote work/outputs/w04_signal_audit.json

Remember to replace the 'FILL IN' verdict with what you actually read.


Two of the three general beliefs tested did not survive, and both failures point the same way:
a content team should not prioritise on keyword search volume or on word count, because
neither is observed to separate declining pages from growing ones in this slice. The one
belief that held — the CTR cliff — is methodological as much as practical: it means any
judgement about whether a page under-captures clicks has to be made against other pages at
the same position, since position alone explains most of the variation.

For the refresh flag specifically: [one sentence stating your Section 3 verdict and what it
implies — if CONFIRMED, that the stale-and-visible combination is a defensible place to spend
review time; if MIXED or FALSE, that the rule needs re-grounding before a team leans on it].

All of this is observed and directional on one 30,000-row anonymized sample with a
same-window proxy label. It is evidence for where to look first, not proof about any
individual page, and nothing here shows that acting on a flagged page causes recovery.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.